# Neuroevolution Hyperparameter Search

This notebook launches multiple neuroevolution runs by calling the project scripts, evaluates the resulting winner genomes on the held-out test split, and compares the outcomes to identify the best genome.

In [1]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "pipeline_config.json").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing config/pipeline_config.json")


def run_command(args: list[str], cwd: Path) -> subprocess.CompletedProcess:

    completed = subprocess.run(
        args,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout.strip())
    if completed.stderr:
        print(completed.stderr.strip())
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with code {completed.returncode}: {printable}")
    return completed


project_root = resolve_project_root(Path.cwd().resolve())
python_exe = Path(sys.executable)

print(f"Project root: {project_root}")
print(f"Python executable: {python_exe}")

Project root: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos
Python executable: c:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\.venv\Scripts\python.exe


## Search Space

The optimization script is varied across sample size, number of generations, and random seed.

The inference configuration remains fixed so that all genomes are compared under the same operational constraints.

In [2]:
search_space = [
    {
        "name": "fast_small_sample",
        "sample_size": 500,
        "generations": 25,
        "random_state": 42,
    },
    {
        "name": "base_line",
        "sample_size": 1000,
        "generations": 50,
        "random_state": 42,
    },
    {
        "name": "larger_sample",
        "sample_size": 1500,
        "generations": 50,
        "random_state": 42,
    },
    {
        "name": "deeper_evolution",
        "sample_size": 1000,
        "generations": 75,
        "random_state": 123,
    },
]

dataset_path = "data/split/dataset_optimization_cereal_co2_train_scaled.csv"
test_split_path = "data/split/dataset_optimization_cereal_co2_test_raw.csv"
neat_config_path = "config/config-feedforward.txt"
results_subdir = "tuning_hyperparameters"
model_dir = project_root / "models/artifacts" / results_subdir
prediction_dir = project_root / "data/predictions" / results_subdir
metrics_dir = project_root / "models/metrics" / results_subdir
model_dir.mkdir(parents=True, exist_ok=True)
prediction_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

inference_caps = {
    "cap_animal_feed": 80.0,
    "cap_composting": 140.0,
    "cap_biochar": 45.0,
    "cap_biomass_combustion": 10000.0,
}

print(pd.DataFrame(search_space))

                name  sample_size  generations  random_state
0  fast_small_sample          500           25            42
1          base_line         1000           50            42
2      larger_sample         1500           50            42
3   deeper_evolution         1000           75           123


## Run Optimization, Inference, and Evaluation

Each scenario writes its winner genome to a unique path, then runs inference on the test split and evaluates the resulting CSV.

In [3]:
records = []

for scenario in search_space:
    scenario_name = scenario["name"]
    winner_path = f"models/artifacts/{results_subdir}/winner_{scenario_name}.pkl"
    prediction_path = f"data/predictions/{results_subdir}/inference_{scenario_name}.csv"
    report_path = f"models/metrics/{results_subdir}/inference_{scenario_name}.json"

    optimization_cmd = [
        str(python_exe),
        "-m",
        "scripts.run_optimization",
        "--project-root",
        ".",
        "--dataset-path",
        dataset_path,
        "--neat-config-path",
        neat_config_path,
        "--sample-size",
        str(scenario["sample_size"]),
        "--generations",
        str(scenario["generations"]),
        "--random-state",
        str(scenario["random_state"]),
        "--winner-output",
        winner_path,
        "--log-level",
        "INFO",
    ]

    inference_cmd = [
        str(python_exe),
        "-m",
        "scripts.run_inference",
        "--project-root",
        ".",
        "--dataset-path",
        test_split_path,
        "--output-path",
        prediction_path,
        "--model-path",
        winner_path,
        "--neat-config-path",
        neat_config_path,
        "--lots-per-day",
        "15",
        "--cap-animal-feed",
        str(inference_caps["cap_animal_feed"]),
        "--cap-composting",
        str(inference_caps["cap_composting"]),
        "--cap-biochar",
        str(inference_caps["cap_biochar"]),
        "--cap-biomass-combustion",
        str(inference_caps["cap_biomass_combustion"]),
        "--log-level",
        "INFO",
    ]

    evaluation_cmd = [
        str(python_exe),
        "-m",
        "scripts.evaluate_inference",
        "--project-root",
        ".",
        "--input-path",
        prediction_path,
        "--report-path",
        report_path,
        "--log-level",
        "INFO",
    ]

    run_command(optimization_cmd, cwd=project_root)
    run_command(inference_cmd, cwd=project_root)
    run_command(evaluation_cmd, cwd=project_root)

    report = json.loads((project_root / report_path).read_text(encoding="utf-8"))
    records.append({
        "scenario": scenario_name,
        "sample_size": scenario["sample_size"],
        "generations": scenario["generations"],
        "random_state": scenario["random_state"],
        "winner_path": winner_path,
        "prediction_path": prediction_path,
        "report_path": report_path,
        "baseline_total_emissions_kg": report["baseline_total_emissions_kg"],
        "ai_estimated_total_emissions_kg": report["ai_estimated_total_emissions_kg"],
        "total_emissions_reduction_pct": report["total_emissions_reduction_pct"],
        "mean_emissions_reduction_pct": report["mean_emissions_reduction_pct"],
        "distribution_ai_pct_normalized": report["distribution_ai_pct_normalized"],
    })

results_df = pd.DataFrame(records).sort_values("total_emissions_reduction_pct", ascending=False).reset_index(drop=True)
display(results_df[[
    "scenario",
    "sample_size",
    "generations",
    "random_state",
    "total_emissions_reduction_pct",
    "mean_emissions_reduction_pct",
    "ai_estimated_total_emissions_kg",
    "baseline_total_emissions_kg",
]])

2026-04-26 10:56:55,254 | INFO | optimization | Loading dataset from C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_train_scaled.csv
2026-04-26 10:56:55,317 | INFO | optimization | Starting neuroevolution for 25 generations with sample_size=500
2026-04-26 10:58:29,139 | INFO | optimization | Evolution finished successfully.
2026-04-26 10:58:29,139 | INFO | optimization | Winning genome saved to C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\models\artifacts\tuning_hyperparameters\winner_fast_small_sample.pkl
2026-04-26 10:58:29,837 | INFO | optimization | Loading dataset from C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_test_raw.csv
2026-04-26 10:58:29,853 | INFO | optimization | Loading preprocessing scaler from C:\Users\IA\Desktop\Proyectos\DATAGI

,scenario,sample_size,generations,random_state,total_emissions_reduction_pct,mean_emissions_reduction_pct,ai_estimated_total_emissions_kg,baseline_total_emissions_kg
0,fast_small_sample,500,25,42,5.74,5.74,40107728.29,42548061.68
1,base_line,1000,50,42,5.74,5.74,40107728.29,42548061.68
2,larger_sample,1500,50,42,5.74,5.74,40107728.29,42548061.68
3,deeper_evolution,1000,75,123,5.74,5.74,40107728.29,42548061.68


## Best Genome

The best genome is the winner from the scenario with the highest CO2 reduction on the held-out test split.

In [4]:
best_row = results_df.iloc[0]
best_summary = pd.DataFrame([best_row[[
    "scenario",
    "sample_size",
    "generations",
    "random_state",
    "winner_path",
    "prediction_path",
    "report_path",
    "total_emissions_reduction_pct",
    "mean_emissions_reduction_pct",
]]])
display(best_summary)

dist_rows = []
for _, row in results_df.iterrows():
    for strategy, pct in row["distribution_ai_pct_normalized"].items():
        dist_rows.append({
            "scenario": row["scenario"],
            "strategy": strategy,
            "assigned_pct": pct,
        })

dist_df = pd.DataFrame(dist_rows)
display(dist_df.sort_values(["scenario", "assigned_pct"], ascending=[True, False]))

,scenario,sample_size,generations,random_state,winner_path,prediction_path,report_path,total_emissions_reduction_pct,mean_emissions_reduction_pct
0,fast_small_sample,500,25,42,models/artifacts/tuning_hyperparameters/winner...,data/predictions/tuning_hyperparameters/infere...,models/metrics/tuning_hyperparameters/inferenc...,5.74,5.74


,scenario,strategy,assigned_pct
4,base_line,composting,46.38
5,base_line,animal_feed,22.49
6,base_line,biomass_combustion,20.11
7,base_line,biochar,11.02
12,deeper_evolution,composting,46.38
13,deeper_evolution,animal_feed,22.49
14,deeper_evolution,biomass_combustion,20.11
15,deeper_evolution,biochar,11.02
0,fast_small_sample,composting,46.38
1,fast_small_sample,animal_feed,22.49
